# Project 4: Decision Trees and Random Forest

## Regardless of the level you are in this course, you are NOT allowed to use import tools from the scipy or sci-kit learn.  The only imports allowed on this project are pandas, numpy, and matplotlib (or plotly/seaborn if you prefer).

### Name: Jonathan Brown and Dyl
### Course Level: 

## Due: Wednesday April 15, 2026

**Introduction:**
* In this project, we explore the application of classification using: a) Decision Tree and b) Random Forest.

**Objectives:**
* The objective of this project is to develop software modules for classification of Titanic passengers via decision trees and random forests

## All Students

* The first problem we aim to us Information Gain to build a decision tree for classification.  The tree will complete when one of two conditions are met:

1. The maximum tree depth is reached.
2. The minimum number of samples is reached in a given leaf node.

**Problem A (70pts)**

1 (5pts). The first thing you'll need to do is read the Titanic dataset from D2L (details about the dataset can be found [Here](https://www.kaggle.com/c/titanic/data?select=test.csv) as illustrated in the Figure below).  Note that you will need both the train.csv (to build your tree) and test.csv to evaluate the classificaiton accuracy.

<img src="Figures/Titanic.png" alt="Titanic Details">

In [ ]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Read in the data (however you prefer) #

badCols = ["Survived", "PassengerId", "Name", "Ticket", "Cabin"]
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# train.plot(kind= 'scatter', x='Age', y='Fare', c='Survived', colormap='viridis')
# train.plot(kind= 'scatter', x='Parch', y='Fare', c='Pclass', colormap='viridis')

for col in train.columns:
    if col == "Survived":
        continue

    if pd.api.types.is_numeric_dtype(train[col]):
        fill_value = train[col].median()
        train[col] = train[col].fillna(fill_value)
        if col in test.columns:
            test[col] = test[col].fillna(fill_value)
    else:
        fill_value = train[col].mode()[0]
        train[col] = train[col].fillna(fill_value)
        if col in test.columns:
            test[col] = test[col].fillna(fill_value)

train_labels = train["Survived"].tolist()
train_data = train.drop(columns=[c for c in badCols if c in train.columns]).values.tolist()

if "Survived" in test.columns:
    test_labels = test["Survived"].tolist()
else:
    test_labels = None

test_data = test.drop(columns=[c for c in badCols if c in test.columns]).values.tolist()

2 (20pts). Next, we need a couple helper functions to compute the conditional entropy and information gain.

**Recall: Entropy** 
$$
    H(X) = -\sum_{\textbf{x} \in X} p(\textbf{x}) \log_2( p(\textbf{x}) )
$$

**Recall: Conditional Entropy** 
$$
    H(Y|X=\textbf{x}) = -\sum_{\textbf{y} \in Y} p(\textbf{y}|\textbf{x}) \log_2 ( p(\textbf{y}|\textbf{x}) )
$$
where
$$
    p(\textbf{y}|\textbf{x}) = \frac{p(\textbf{y},\textbf{x})}{p(\textbf{x})}
$$
and
$$
    p(\textbf{x}) = \sum_{\textbf{y} \in Y} p(\textbf{y},\textbf{x})
$$

**Recall: Information Gain** 
$$
    IG(Y|X) = H(Y) - H(Y|X)
$$



In [20]:
# Function definitions here (you may want to modify the input parameters depending on your implimentation #
def count( d_X, x ):
    countX = 0
    for item in d_X:
        if item == x:
            countX += 1
    return countX

def condProb(d_X, d_Y, x, y):
    countX = 0
    countXY = 0
    for i in range (len(d_X)):
        if d_X[i] == x:
            countX += 1
            if d_Y[i] == y:
                countXY += 1
    if countX == 0:
        return 0
    return countXY / countX

def Entropy(d_X):
    H = 0
    totalCount = len(d_X)
    for x in set(d_X):
        count_x = count( d_X, x)
        p_X = count_x / totalCount
        if p_X > 0:
            H += p_X * np.log2(p_X)
    return -H

def C_Entropy(d_X,d_Y):
    cH = 0
    totalCount = len( d_X )
    for x in set(d_X):
        count_x = count( d_X, x)
        p_X = count_x / totalCount
        sum_y = 0
        for y in set(d_Y[i] for i in range(totalCount) if d_X[i] == x):
            event = condProb( d_X, d_Y, x, y)
            if event > 0:
                sum_y += event * np.log2( event )
        cH += p_X * sum_y
    return -cH

def IG(d_X,d_Y):
    ig = Entropy( d_Y ) - C_Entropy( d_X, d_Y ) 
    return ig

def is_number_column(column):
    try:
        for x in column:
            float(x)
        return True
    except:
        return False


def IG_numeric(column, l_labels):
    values = sorted(set(float(x) for x in column))
    if len(values) <= 1:
        return -1, None

    bestIG = -1
    bestThreshold = None
    parent_entropy = Entropy(l_labels)

    for i in range(len(values) - 1):
        threshold = (values[i] + values[i + 1]) / 2.0

        left_labels = []
        right_labels = []

        for j in range(len(column)):
            if float(column[j]) <= threshold:
                left_labels.append(l_labels[j])
            else:
                right_labels.append(l_labels[j])

        if len(left_labels) == 0 or len(right_labels) == 0:
            continue

        cH = (len(left_labels) / len(column)) * Entropy(left_labels) + \
             (len(right_labels) / len(column)) * Entropy(right_labels)

        ig = parent_entropy - cH

        if ig > bestIG:
            bestIG = ig
            bestThreshold = threshold

    return bestIG, bestThreshold

3 (30pts). Define a function to build the tree, recall there should be one of two exit criterion:

1. The maximum tree depth is reached
2. The minimum number of samples is reached in a given leaf node.


In [21]:
# Build Tree (can be recursive) #
    
def common_label( l_labels ):
    counts = {}
    for label in l_labels:
        if label not in counts:
            counts[label] = 0
        counts[label] += 1

    best_label = None
    best_count = -1

    for label in counts:
        if counts[label] > best_count:
            best_count = counts[label]
            best_label = label

    return best_label
    
def BuildTree(d_data,l_labels):
    if len(set(l_labels)) == 1:
        return l_labels[0]

    # No features left
    if len(d_data[0]) == 0:
        return common_label(l_labels)

    # Pick best information gain
    bestIG = -1
    bestFeature = -1
    bestThreshold = None
    bestIsNumber = False

    for i in range(len(d_data[0])):
        column = [row[i] for row in d_data]

        if is_number_column(column):
            ig, threshold = IG_numeric(column, l_labels)
            if ig > bestIG:
                bestFeature = i
                bestIG = ig
                bestThreshold = threshold
                bestIsNumber = True
        else:
            ig = IG(column, l_labels)
            if ig > bestIG:
                bestFeature = i
                bestIG = ig
                bestThreshold = None
                bestIsNumber = False

    if bestFeature == -1:
        return common_label(l_labels)

    # Create node
    tree = {}
    tree["feature"] = bestFeature
    tree["children"] = {}
    tree["label"] = common_label(l_labels)
    tree["numeric"] = bestIsNumber

    if bestIsNumber:
        tree["threshold"] = bestThreshold

        left_data = []
        left_labels = []
        right_data = []
        right_labels = []

        for i in range(len(d_data)):
            new_row = d_data[i][:bestFeature] + d_data[i][bestFeature + 1:]

            if float(d_data[i][bestFeature]) <= bestThreshold:
                left_data.append(new_row)
                left_labels.append(l_labels[i])
            else:
                right_data.append(new_row)
                right_labels.append(l_labels[i])

        if len(left_data) == 0:
            tree["children"]["<="] = common_label(l_labels)
        else:
            tree["children"]["<="] = BuildTree(left_data, left_labels)

        if len(right_data) == 0:
            tree["children"][">"] = common_label(l_labels)
        else:
            tree["children"][">"] = BuildTree(right_data, right_labels)

    else:
        values = set(row[bestFeature] for row in d_data)

        for v in values:
            subset_data = []
            subset_labels = []

            for i in range(len(d_data)):
                if d_data[i][bestFeature] == v:
                    # remove the used feature
                    new_row = d_data[i][:bestFeature] + d_data[i][bestFeature + 1:]
                    subset_data.append(new_row)
                    subset_labels.append(l_labels[i])

            if len(subset_data) == 0:
                tree["children"][v] = common_label(l_labels)
            else:
                tree["children"][v] = BuildTree(subset_data, subset_labels)

    return tree

4 (15pts). Next write a function to perform the prediction (classification) of a new training sample:

In [22]:
# Function to predict #
def DT_predict(d_tree,d_sample):
    # Case 1: Leaf node
    if not isinstance(d_tree, dict):
        return d_tree

    feature = d_tree["feature"]

    if d_tree.get("numeric", False):
        value = float(d_sample[feature])

        # remove the feature before passing down
        new_sample = d_sample[:feature] + d_sample[feature + 1:]

        if value <= d_tree["threshold"]:
            return DT_predict(d_tree["children"]["<="], new_sample)
        else:
            return DT_predict(d_tree["children"][">"], new_sample)

    value = d_sample[feature]

    if value not in d_tree["children"]:
        return d_tree["label"]

    # remove the feature before passing down
    new_sample = d_sample[:feature] + d_sample[feature + 1:]

    return DT_predict(d_tree["children"][value], new_sample)

5 (5pts). Using the test set, evaluate the accuracy of your classification tree.

In [24]:
# Classification Accuracy #
def accuracy(t_test, t_tree, l_labels):
    correct = 0
    total = len(t_test)
    for i in range( total ):
        prediction = DT_predict(t_tree, t_test[i])
        actual = l_labels[i]
        if ( prediction == actual):
            correct += 1

    a_accuracy = correct / total
    return a_accuracy

tree = BuildTree(train_data, train_labels)
acc = accuracy( test_data, tree, test_labels)
print(acc)

0.7703349282296651


**Problem B (30pts)**
* In Problem A, you designed a function to build a decision tree.  In this problem, you will use that function to build several different trees via bagging (bootstrapping and aggregation).  The result will be a random forest.  You should **at minimum** have a forest of at least 5 trees. 

* Once the forest is built, similar to A.5, design a function for comparing the accuracy of a standard tree, vs. random forest on the testing dataset.



In [ ]:
# Build Random Forest #
# prepare data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train = train.to_numpy()
test = test.to_numpy()

rng = np.random.default_rng(seed=448)
# choose an odd num because the class decision is handled by np.argmax
TREE_COUNT = 5
rand_trees = []

# BOOTSTRAPPING
for i in range(TREE_COUNT):
    # select random sample indexes w/replacement
    r_samp_idxs = rng.choice(train.shape[0], size=train.shape[0], replace=True)

    # select random features indexes
    # number of features = 3 (sqrt of the number of features(10))
    r_feature_idxs = rng.integers(2, 11, size=3, endpoint=True)
    #print(r_feature_idxs)

    tree_data = train[np.ix_(r_samp_idxs, r_feature_idxs)]
    l_labels = train[np.ix_(r_samp_idxs, [1])]
    rand_trees.append(BuildTree(tree_data, l_labels)) # build a random tree

# AGGREGATION
# grab random samples w/ replacement to classify
r_samp_idxs = rng.choice(test.shape[0], size=test.shape[0], replace=True)

best_choices = []
for i in range(test.shape[0]):
    # collect answers from each random tree
    agg = []
    for j in range(TREE_COUNT):
        agg.append(DT_predict(rand_trees[j], test[r_samp_idxs[i]]))
    # choose the most frequent class amongst the random tree answers
    best_choices[i] = np.bincount(agg).argmax()

(891,)


# Because this is a partner project, you MUST list below who did what on the project to ensure equal contribution:

### Jonathan Brown:
1. Problem A

### Dyl:
1. Problem B
2. Solved world hunger

## CSC 548 Only! - You get a Break - No extra work for you!